# Faz 2 - Ön İşleme: Bölme, Ölçekleme ve Pencereleme

Bu notebook, 15 dk ızgaradan sızıntısız train/val/test bölmesini, train-only ölçeklemeyi ve 96 adım girdi / 16 adım çıktı pencerelerini özetler. Mantık `src/` altındadır.

In [1]:
import sys, json, subprocess
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, joblib
from src import config, data, windows
config.apply_plot_style()
print(config.summary())

Yıllar: 2015-2016
Ham parquet: C:\Users\user\Desktop\merve\data\raw\pvdaq_sys4_2015_2016.parquet
Bölme: train 1 segment, val 6 segment, test 6 segment; veri sonu 2016-12-13 00:00
Hedef: ac_power [W]
Kapasite: 1.0 kW = 1000.0 W (nRMSE paydası: CAPACITY_W)


## 1. Izgara
15 dk, right-label ızgara yüklenir ve veri sözleşmesi doğrulanır.

In [2]:
grid = data.load_grid()
print(data.validate_grid(grid))
print(grid.index.min(), '->', grid.index.max())

{'n_rows': 68352, 'nan_ratio': {'ac_power': 0.0, 'poa_irradiance': 0.0, 'ambient_temp': 0.0, 'module_temp_1': 0.0}}
2015-01-01 00:15:00 -> 2016-12-13 00:00:00


## 2. Bölme
Bölme bin başlangıcına göre yapılır: train 2015, val 2016 tek aylar, test 2016 çift aylar. Veri `DATA_END` = 2016-12-13 00:00'da kesilmiştir (Aralık test segmenti 12 gün).

In [3]:
split, seg = windows.assign_segments(grid.index)
tbl = pd.DataFrame({'split': split, 'segment': seg}).groupby(['split','segment']).size().rename('adım')
print(tbl.to_string())
print(config.DATA_END_REASON)

split  segment
test   7           2784
       8           2880
       9           2880
       10          2976
       11          2976
       12          1152
train  0          35040
val    1           2976
       2           2976
       3           2976
       4           2976
       5           2880
       6           2880
2016-12-13 10:13'ten itibaren module_temp_1 sürekli eksik ve 2016-12-14'ten itibaren invertör kapalı (POA 1000+ W/m² iken AC ≈ 3 W); bu günler fiziksel olarak tahmin edilemez. Kullanıcı kararıyla veri 2016-12-13 00:00'da kesildi; Aralık test segmenti 12 gündür.


## 3. Ölçekleyiciler (yalnızca train)
Girdi (9 kanal) ve hedef için ayrı `StandardScaler`, yalnızca train ızgara satırlarına fit edilir.

In [4]:
sc = joblib.load(config.SCALERS_FILE)  # yalnızca kendi ürettiğimiz dosya
print(pd.DataFrame({'ortalama': sc['x'].mean_, 'std': sc['x'].scale_}, index=sc['features']).round(3))
print('hedef (W):', sc['y'].mean_, sc['y'].scale_)
print('n_train_rows =', sc['n_train_rows'])

                ortalama      std
ac_power         164.008  263.220
poa_irradiance   216.810  335.886
ambient_temp      11.566   10.210
module_temp_1     15.550   15.138
hour_sin          -0.000    0.707
hour_cos          -0.000    0.707
doy_sin           -0.000    0.707
doy_cos           -0.001    0.707
sun_elevation      0.308   36.261
hedef (W): [164.00796466] [263.2204006]
n_train_rows = 35040


## 4. Pencereler

In [5]:
for s in ('train','val','test'):
    w = windows.load_windows(s)
    print(s, 'X', w['X'].shape, 'y', w['y'].shape, 'tamamı-gece:', int((~w['y_day'].any(axis=1)).sum()))
w = windows.load_windows('test')
print({config.SEASON_NAMES[k]: int((w['season']==k).sum()) for k in range(4)})
print(open(config.WINDOWS_SUMMARY, encoding='utf-8').read()[:600])

train X (21581, 96, 9) y (21581, 16) tamamı-gece: 0


val

 X (10608, 96, 9) y (10608, 16) tamamı-gece: 0


test X (14982, 96, 9) y (14982, 16) tamamı-gece: 5387
{'Kış': 3714, 'İlkbahar': 2769, 'Yaz': 5634, 'Sonbahar': 2865}
{
  "train": {
    "n_windows": 21581,
    "per_segment": {
      "0": 21581
    },
    "dropped_nan": 0,
    "dropped_all_night": 13348,
    "all_night_windows_kept": 0
  },
  "val": {
    "n_windows": 10608,
    "per_segment": {
      "1": 1515,
      "2": 1783,
      "3": 2015,
      "4": 2038,
      "5": 1756,
      "6": 1501
    },
    "dropped_nan": 0,
    "dropped_all_night": 6390,
    "all_night_windows_kept": 0
  },
  "test": {
    "n_windows": 14982,
    "per_segment": {
      "7": 2673,
      "8": 2769,
      "9": 2769,
      "10": 2865,
      "11": 2865,
      "12": 1041
    },
   


## 5. Birim testi
Şekil, hizalama, segment sınırı, NaN atma, gece filtresi, right-label resample ve train-only scaler.

In [6]:
r = subprocess.run([sys.executable, '-I', str(PROJECT_ROOT/'tests'/'test_windows.py')], capture_output=True, text=True, encoding='utf-8', cwd=PROJECT_ROOT)
print(r.stdout); print(r.stderr[-500:])

OK test_sekil
OK test_hizalama
OK test_segment_siniri
OK test_nan_atma
OK test_gece_filtresi
OK test_resample_sizinti
OK test_resample_min_gecerli_dakika
OK test_train_only_scaler
OK test_fill_kisa_bosluk
OK test_segment_ay_siniri
TESTLER_TAMAM




## 6. Özet
Problem tanımı: `docs/PROBLEM_TANIMI.md`. Hedef AC güç (W), ana ufuk +1 saat (4. adım), gelecek ışınım verisi kullanılmaz (gerçek tahmin, nowcasting değil).